In [12]:
!pip install gensim scikit-learn nltk

import pandas as pd
import numpy as np
import nltk
import re

from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [8]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Natural Language Processing/reviews_dataset.csv")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                                              review     label
0  The product quality is excellent and exceeded ...  Positive
1      I am very satisfied with the service provided  Positive
2  The experience was good and I would recommend ...  Positive
3      Amazing performance and great value for money  Positive
4        Customer support was helpful and responsive  Positive


In [10]:
# Remove missing values
df = df.dropna(subset=['review', 'label'])

# Convert to string (safety)
df['review'] = df['review'].astype(str)
df['label'] = df['label'].astype(str)

print(df.isnull().sum())

review    0
label     0
dtype: int64


In [13]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation
    tokens = word_tokenize(text)
    return tokens

df['tokens'] = df['review'].apply(preprocess)

print(df[['review', 'tokens']].head())

                                              review  \
0  The product quality is excellent and exceeded ...   
1      I am very satisfied with the service provided   
2  The experience was good and I would recommend ...   
3      Amazing performance and great value for money   
4        Customer support was helpful and responsive   

                                              tokens  
0  [the, product, quality, is, excellent, and, ex...  
1  [i, am, very, satisfied, with, the, service, p...  
2  [the, experience, was, good, and, i, would, re...  
3  [amazing, performance, and, great, value, for,...  
4  [customer, support, was, helpful, and, respons...  


In [14]:
w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [15]:
def sentence_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [16]:
X = np.array([sentence_vector(tokens, w2v_model) for tokens in df['tokens']])

In [17]:
le = LabelEncoder()
y = le.fit_transform(df['label'])

print("Classes:", le.classes_)

Classes: ['Negative' 'Neutral' 'Positive']


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
model = LogisticRegression(max_iter=300)
model.fit(X_train, y_train)

LogisticRegression(max_iter=300)

In [20]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.3684210526315789

Classification Report:

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         6
     Neutral       0.00      0.00      0.00         6
    Positive       0.37      1.00      0.54         7

    accuracy                           0.37        19
   macro avg       0.12      0.33      0.18        19
weighted avg       0.14      0.37      0.20        19



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [21]:
def predict_review(text):
    tokens = preprocess(text)
    vec = sentence_vector(tokens, w2v_model).reshape(1, -1)
    pred = model.predict(vec)
    return le.inverse_transform(pred)[0]

# Test
print(predict_review("The product is amazing"))
print(predict_review("Worst experience ever"))

Positive
Positive
